# LeetCode #40 — Combination Sum II: Backtracking khi mảng có phần tử trùng

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def combinationSum2(self, candidates: List[int], target: int) -> List[List[int]]:
```

So với #39, có hai thay đổi:

1. Mỗi phần tử chỉ được dùng **một lần** → đệ quy với `i + 1`.
2. `candidates` **có thể có giá trị trùng** → dễ sinh tổ hợp trùng như `[1, 7]` hai lần (từ hai số `1` khác nhau).

Mục tiêu của notebook là nắm chắc quy tắc bỏ trùng **"cùng tầng thì bỏ, khác tầng thì giữ"**:

```python
if i > start and candidates[i] == candidates[i - 1]:
    continue
```

— một dòng xuất hiện ở rất nhiều bài backtracking (#47, #90, …).

# 1. Bài toán là gì?

Cho `candidates` (có thể chứa số trùng) và `target`. Tìm mọi tổ hợp **không trùng** có tổng bằng `target`. Mỗi **phần tử** (theo vị trí) chỉ dùng **tối đa một lần**.

**Constraints (theo đề):**

- `1 <= candidates.length <= 100`
- `1 <= candidates[i] <= 50`
- `1 <= target <= 30`

---

## Ví dụ

```text
candidates = [10, 1, 2, 7, 6, 1, 5], target = 8
-> [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]

candidates = [2, 5, 2, 1, 2], target = 5
-> [[1, 2, 2], [5]]
```

Ở ví dụ 1, `[1, 1, 6]` dùng **cả hai** số `1` — hợp lệ vì đó là hai phần tử khác nhau. Nhưng `[1, 7]` chỉ được xuất hiện **một lần** trong kết quả, dù có hai cách chọn số `1`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Quay lui (backtracking)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [4]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [5]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [6]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [7]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [8]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Áp dụng vào bài này

Như #39 nhưng mỗi số dùng **một lần** (`start = i + 1`) và đầu vào có **trùng** → **sắp xếp** rồi bỏ trùng **cùng tầng** (`i > start and nums[i] == nums[i−1]`) — đúng mẫu bỏ trùng của quay lui ở trên.

# 3. Thử dùng lại code #39, chỉ đổi `i` thành `i + 1`

In [9]:
from typing import List

def attempt(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

print(attempt([10, 1, 2, 7, 6, 1, 5], 8))

[[1, 1, 6], [1, 2, 5], [1, 7], [1, 2, 5], [1, 7], [2, 6]]


`[1, 2, 5]`, `[1, 7]` xuất hiện **hai lần**: một lần dùng số `1` ở vị trí 0, một lần dùng số `1` ở vị trí 1 (sau khi sắp xếp: `[1, 1, 2, 5, 6, 7, 10]`).

Bỏ trùng bằng `set` ở cuối thì được, nhưng nếu input là `[1] * 30` thì số nhánh trùng bùng nổ theo hàm mũ.

# 4. Quy tắc bỏ trùng: cùng tầng thì bỏ

Sau khi **sắp xếp**, các số bằng nhau đứng cạnh nhau. Xét một tầng của cây (một lần gọi `backtrack(start, ...)`), vòng `for` thử lần lượt `candidates[start], candidates[start+1], ...` làm **phần tử tiếp theo** của `path`.

- Nếu `candidates[i] == candidates[i - 1]` và cả hai đều đang được thử **ở cùng tầng này**, thì nhánh `i` sinh ra **y hệt** các tổ hợp mà nhánh `i - 1` đã sinh (vì nhánh `i - 1` còn được dùng thêm cả `candidates[i]` phía sau). → **Bỏ nhánh `i`**.
- Nhưng ở **tầng sâu hơn** (sau khi đã chọn `candidates[i-1]`), việc chọn tiếp `candidates[i]` là hợp lệ — đó là tổ hợp có **hai** số bằng nhau như `[1, 1, 6]`.

Điều kiện phân biệt hai trường hợp: `i > start`.

- `i == start`: phần tử đầu tiên của tầng → luôn được thử.
- `i > start` và bằng phần tử trước → trùng **cùng tầng** → bỏ.

```python
for i in range(start, len(candidates)):
    if i > start and candidates[i] == candidates[i - 1]:
        continue
```

## Hình dung với `[1, 1, 2]`, `target = 3`

```text
                 tầng 0 (start=0): thử 1(i=0), 1(i=1)✗ bỏ, 2(i=2)
                /                                    \
     path=[1] tầng 1 (start=1):                      path=[2] ...
     thử 1(i=1) ✓ GIỮ (i == start), 2(i=2)
         |                  |
     path=[1,1]          path=[1,2] ✓ tổng 3
     tầng 2: thử 2 -> tổng 4 > 3
```

- Ở tầng 0: số `1` thứ hai (`i=1`) bị bỏ vì trùng **cùng tầng** với `i=0`.
- Ở tầng 1: số `1` thứ hai (`i=1 == start`) được **giữ** → cho phép tổ hợp `[1, 1, ...]`.

In [10]:
def combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue                        # trùng cùng tầng -> bỏ
            if candidates[i] > remain:
                break                           # đã sắp xếp -> các số sau còn lớn hơn
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])   # mỗi phần tử dùng một lần
            path.pop()

    backtrack(0, target)
    return result

print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))
print(combination_sum2([2, 5, 2, 1, 2], 5))

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
[[1, 2, 2], [5]]


# 5. Mô phỏng: thấy các nhánh bị bỏ

In [11]:
def trace_combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    print("Sau sắp xếp:", candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        indent = "  " * len(path)
        if remain == 0:
            print(f"{indent}✓ ghi {path}")
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            x = candidates[i]
            if i > start and x == candidates[i - 1]:
                print(f"{indent}bỏ {x} (i={i}): trùng cùng tầng với i={i - 1}")
                continue
            if x > remain:
                print(f"{indent}{x} > {remain}: break")
                break
            path.append(x)
            print(f"{indent}chọn {x} (i={i}) -> {path}, còn {remain - x}")
            backtrack(i + 1, remain - x)
            path.pop()

    backtrack(0, target)
    return result

print("Kết quả:", trace_combination_sum2([2, 5, 2, 1, 2], 5))

Sau sắp xếp: [1, 2, 2, 2, 5]
chọn 1 (i=0) -> [1], còn 4
  chọn 2 (i=1) -> [1, 2], còn 2
    chọn 2 (i=2) -> [1, 2, 2], còn 0
      ✓ ghi [1, 2, 2]
    bỏ 2 (i=3): trùng cùng tầng với i=2
    5 > 2: break
  bỏ 2 (i=2): trùng cùng tầng với i=1
  bỏ 2 (i=3): trùng cùng tầng với i=2
  5 > 4: break
chọn 2 (i=1) -> [2], còn 3
  chọn 2 (i=2) -> [2, 2], còn 1
    2 > 1: break
  bỏ 2 (i=3): trùng cùng tầng với i=2
  5 > 3: break
bỏ 2 (i=2): trùng cùng tầng với i=1
bỏ 2 (i=3): trùng cùng tầng với i=2
chọn 5 (i=4) -> [5], còn 0
  ✓ ghi [5]
Kết quả: [[1, 2, 2], [5]]


Quan sát ở tầng 1 sau khi chọn `1`: thử `2` (i=1, `== start` → giữ), rồi ở tầng 2 thử `2` (i=2, `== start` → giữ) → `[1, 2, 2]`. Nhưng cũng ở tầng 1, `2` ở `i=2` và `i=3` bị bỏ vì trùng cùng tầng.

# 6. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def combinationSum2(self, candidates: List[int], target: int) -> List[List[int]]:
        candidates.sort()
        result, path = [], []

        def backtrack(start: int, remain: int):
            if remain == 0:
                result.append(path[:])
                return
            for i in range(start, len(candidates)):
                if i > start and candidates[i] == candidates[i - 1]:
                    continue
                if candidates[i] > remain:
                    break
                path.append(candidates[i])
                backtrack(i + 1, remain - candidates[i])
                path.pop()

        backtrack(0, target)
        return result

In [13]:
def normalize(combos):
    return sorted(sorted(c) for c in combos)

def check(name, actual, expected):
    assert normalize(actual) == normalize(expected), (name, actual, expected)
    assert len(actual) == len({tuple(sorted(c)) for c in actual}), (name, "có trùng", actual)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.combinationSum2([10, 1, 2, 7, 6, 1, 5], 8), [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]])
check("Ví dụ 2", sol.combinationSum2([2, 5, 2, 1, 2], 5), [[1, 2, 2], [5]])
check("Không có lời giải", sol.combinationSum2([3, 4], 2), [])
check("Toàn số giống nhau", sol.combinationSum2([1] * 10, 3), [[1, 1, 1]])
check("Dùng hết mảng", sol.combinationSum2([1, 2, 3], 6), [[1, 2, 3]])

Ví dụ 1                    PASS  [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
Ví dụ 2                    PASS  [[1, 2, 2], [5]]
Không có lời giải          PASS  []
Toàn số giống nhau         PASS  [[1, 1, 1]]
Dùng hết mảng              PASS  [[1, 2, 3]]


## Đối chiếu với cách sinh tập con rồi bỏ trùng

Mảng nhỏ → liệt kê mọi tập con (theo vị trí) bằng `itertools.combinations`, giữ những tập có tổng đúng, bỏ trùng bằng set.

In [14]:
import random
from itertools import combinations

def bruteforce(candidates, target):
    found = set()
    for k in range(1, len(candidates) + 1):
        for combo in combinations(candidates, k):
            if sum(combo) == target:
                found.add(tuple(sorted(combo)))
    return [list(t) for t in found]

random.seed(40)
for _ in range(1500):
    cands = [random.randint(1, 6) for _ in range(random.randint(1, 10))]
    t = random.randint(1, 12)
    assert normalize(sol.combinationSum2(cands[:], t)) == normalize(bruteforce(cands, t)), (cands, t)
print("1500 test ngẫu nhiên: khớp brute force")

1500 test ngẫu nhiên: khớp brute force


# 7. Vì sao đúng? Độ phức tạp?

**Không bỏ sót:** mỗi tổ hợp kết quả (một **multiset** giá trị) có một cách chọn "chuẩn": với mỗi giá trị lặp `m` lần, dùng **`m` bản đầu tiên** của giá trị đó trong mảng đã sắp xếp. Cách chọn chuẩn này không bao giờ vi phạm quy tắc bỏ trùng (mỗi lần chọn bản kế tiếp của cùng giá trị, nó là phần tử **đầu** của tầng: `i == start`). Vậy nó có trong cây.

**Không trùng:** mọi cách chọn khác của cùng multiset đều phải "nhảy cóc" một bản sao (chọn bản `i` khi bản `i-1` cùng giá trị chưa được chọn ở cùng tầng) → bị `continue` chặn.

**Độ phức tạp:** mỗi phần tử chọn hoặc không → tối đa `2^n` tập con; cắt tỉa và bỏ trùng giảm mạnh trên thực tế.

```text
Time:  O(2^n · n)   (chặn trên; thực tế nhỏ hơn nhiều vì target <= 30)
Space: O(n) cho path và đệ quy
```

# 8. Những lỗi dễ mắc

## Lỗi 1: Bỏ trùng bằng `i > 0` thay vì `i > start`

Bỏ luôn cả trường hợp **khác tầng** → mất tổ hợp có số lặp như `[1, 1, 6]`.

In [15]:
def wrong_i_gt_0(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > 0 and candidates[i] == candidates[i - 1]:     # SAI
                continue
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

print("Sai :", wrong_i_gt_0([10, 1, 2, 7, 6, 1, 5], 8))
print("Đúng:", combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))

Sai : [[1, 2, 5], [1, 7], [2, 6]]
Đúng: [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


## Lỗi 2: Quên sắp xếp

Quy tắc bỏ trùng dựa trên việc **số bằng nhau nằm cạnh nhau**. Không sắp xếp → `[1, 2, 1]` không phát hiện được trùng.

In [16]:
def wrong_unsorted(candidates: List[int], target: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remain:
                continue
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])
            path.pop()

    backtrack(0, target)                       # thiếu sort
    return result

print("Sai :", wrong_unsorted([1, 2, 1], 3))
print("Đúng:", combination_sum2([1, 2, 1], 3))

Sai :

 [[1, 2], [2, 1]]
Đúng: [[1, 2]]


## Lỗi 3: Đặt kiểm tra trùng **sau** `break`

Thứ tự không ảnh hưởng tính đúng ở đây, nhưng hãy giữ thói quen: kiểm tra trùng trước, rồi mới cắt tỉa.

## Lỗi 4: Đệ quy với `i` (như #39)

Một phần tử bị dùng nhiều lần → `[1, 1, 1, ...]` dù mảng chỉ có một số `1`.

# 9. Mẹo nhận diện pattern

Khi input **có giá trị trùng** và kết quả **không được trùng**:

1. **Sắp xếp**.
2. Trong vòng `for` của backtracking: `if i > start and a[i] == a[i - 1]: continue`.

Gặp ở: **#40**, **#90 Subsets II**, **#47 Permutations II** (dạng khác: dùng `used[i-1]`), **#15 3Sum** (dạng hai con trỏ).

## Template

```python
a.sort()
def backtrack(start, remain):
    if remain == 0: ghi path[:]; return
    for i in range(start, len(a)):
        if i > start and a[i] == a[i - 1]: continue    # trùng cùng tầng
        if a[i] > remain: break                         # cắt tỉa
        path.append(a[i]); backtrack(i + 1, remain - a[i]); path.pop()
```

# 10. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `candidates = [1, 1, 1, 2]`, `target = 3`, liệt kê các nhánh bị bỏ do trùng và kết quả cuối.

## Bài 2 — Điền chỗ trống

In [17]:
def exercise_2_combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            # TODO 1: bỏ trùng cùng tầng
            # TODO 2: cắt tỉa khi candidates[i] > remain
            # TODO 3: chọn, đệ quy với i + 1, bỏ chọn
            pass

    backtrack(0, target)
    return result


# assert normalize(exercise_2_combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
# assert normalize(exercise_2_combination_sum2([2, 5, 2, 1, 2], 5)) == [[1, 2, 2], [5]]

## Bài 3 — Biến thể: Subsets (LeetCode #78)

Liệt kê **mọi tập con** của mảng gồm các phần tử **khác nhau** (kể cả tập rỗng).

```python
subsets([1, 2, 3]) -> [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]   (thứ tự tùy ý)
```

Gợi ý nhanh: ghi `path` **ở mọi nút** (không chỉ ở lá).

In [18]:
def exercise_3_subsets(nums: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_3_subsets([1, 2, 3])) == normalize([[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]])
# assert exercise_3_subsets([]) == [[]]

## Bài 4 — Biến thể: Subsets II (LeetCode #90)

Như Bài 3 nhưng mảng **có thể có trùng**; kết quả không được trùng.

```python
subsets_with_dup([1, 2, 2]) -> [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
```

In [19]:
def exercise_4_subsets_with_dup(nums: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_4_subsets_with_dup([1, 2, 2])) == normalize([[], [1], [1, 2], [1, 2, 2], [2], [2, 2]])
# assert exercise_4_subsets_with_dup([0]) == [[], [0]]

## Bài 5 — Mini challenge: tổ hợp có đúng `k` phần tử

Như bài chính, nhưng chỉ lấy tổ hợp có **đúng `k` phần tử**.

```python
combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 2) -> [[1, 7], [2, 6]]
combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 3) -> [[1, 1, 6], [1, 2, 5]]
```

Hãy cắt tỉa thêm: dừng đi sâu khi `len(path) == k`.

In [20]:
def exercise_5_combination_sum2_k(candidates: List[int], target: int, k: int) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_5_combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 2)) == [[1, 7], [2, 6]]
# assert normalize(exercise_5_combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 3)) == [[1, 1, 6], [1, 2, 5]]

# 11. Gợi ý

**Bài 1**
- Mức 1: ở tầng 0, chỉ số `1` đầu tiên được thử.
- Mức 2: kết quả gồm `[1, 1, 1]` và `[1, 2]`.

**Bài 2**
- Mức 1: so với code mục 6.

**Bài 3**
- Mức 1: `result.append(path[:])` ở **đầu** hàm `backtrack(start)`, không cần điều kiện.
- Mức 2: vòng `for i in range(start, n)`, đệ quy `i + 1`.

**Bài 4**
- Mức 1: Bài 3 + sắp xếp + bỏ trùng cùng tầng.

**Bài 5**
- Mức 1: base case: `if len(path) == k: if remain == 0: ghi; return`.

# 12. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
print("Kết quả:", trace_combination_sum2([1, 1, 1, 2], 3))

Sau sắp xếp: [1, 1, 1, 2]
chọn 1 (i=0) -> [1], còn 2
  chọn 1 (i=1) -> [1, 1], còn 1
    chọn 1 (i=2) -> [1, 1, 1], còn 0
      ✓ ghi [1, 1, 1]
    2 > 1: break
  bỏ 1 (i=2): trùng cùng tầng với i=1
  chọn 2 (i=3) -> [1, 2], còn 0
    ✓ ghi [1, 2]
bỏ 1 (i=1): trùng cùng tầng với i=0
bỏ 1 (i=2): trùng cùng tầng với i=1
chọn 2 (i=3) -> [2], còn 1
Kết quả: [[1, 1, 1], [1, 2]]


## Đáp án Bài 2

In [22]:
def answer_2_combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

assert normalize(answer_2_combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
assert normalize(answer_2_combination_sum2([2, 5, 2, 1, 2], 5)) == [[1, 2, 2], [5]]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [23]:
def answer_3_subsets(nums: List[int]) -> List[List[int]]:
    result, path = [], []

    def backtrack(start: int):
        result.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return result

assert normalize(answer_3_subsets([1, 2, 3])) == normalize([[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]])
assert answer_3_subsets([]) == [[]]
print("Bài 3 PASS:", answer_3_subsets([1, 2, 3]))

Bài 3 PASS: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


## Đáp án Bài 4

In [24]:
def answer_4_subsets_with_dup(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result, path = [], []

    def backtrack(start: int):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return result

assert normalize(answer_4_subsets_with_dup([1, 2, 2])) == normalize([[], [1], [1, 2], [1, 2, 2], [2], [2, 2]])
assert answer_4_subsets_with_dup([0]) == [[], [0]]
print("Bài 4 PASS:", answer_4_subsets_with_dup([1, 2, 2]))

Bài 4 PASS: [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


## Đáp án Bài 5

In [25]:
def answer_5_combination_sum2_k(candidates: List[int], target: int, k: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if len(path) == k:
            if remain == 0:
                result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

assert normalize(answer_5_combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 2)) == [[1, 7], [2, 6]]
assert normalize(answer_5_combination_sum2_k([10, 1, 2, 7, 6, 1, 5], 8, 3)) == [[1, 1, 6], [1, 2, 5]]
print("Bài 5 PASS")

Bài 5 PASS


# 13. Tổng kết

> Mỗi phần tử một lần → đệ quy `i + 1`. Có trùng → **sắp xếp** + `if i > start and a[i] == a[i-1]: continue` (bỏ trùng **cùng tầng**, giữ trùng **khác tầng**).

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp?
- [ ] Bỏ trùng bằng `i > start` (không phải `i > 0`)?
- [ ] Đệ quy với `i + 1`?
- [ ] Cắt tỉa `break` khi `candidates[i] > remain`?
- [ ] Đã thử mảng toàn số giống nhau?